In [4]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_deepseek import ChatDeepSeek
import os
from dotenv import load_dotenv
load_dotenv(override=True)
#0. 连接LLM模型
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = ChatDeepSeek(
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    model_name="deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)
#1. 定义状态
class OverAllState(MessagesState):
    username: str
    output: str
#2. 定义节点
def node_a(state:OverAllState)->OverAllState:
    return{
        "messages":[HumanMessage("你好,我是"+state["username"])]
    }

def llm_node(state:OverAllState)->OverAllState:
    res=model.invoke(state["messages"])
    return{
        "messages":[res],
        "output":res.content
    }
#3. 构建图
builder=StateGraph(state_schema=OverAllState)
builder.add_node("node_a",node_a)
builder.add_node("llm_node",llm_node)
builder.add_edge(START,"node_a")
builder.add_edge("node_a","llm_node")
builder.add_edge("llm_node",END)

graph=builder.compile()
#4. 运行图
result=graph.invoke({"username":"老王"})
print(result)

{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='cba0702a-d52b-432b-9887-c7d3b6fac99f'), AIMessage(content='你好，老王！很高兴认识你。👋\n\n有什么我可以帮你的吗？无论是聊天、解答问题、帮忙写东西，还是别的什么，尽管说～', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 8, 'total_tokens': 42, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'ca3294b5-3040-4812-9f87-cac305cb2c64', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0bdcc-1d7a-7270-bbf0-84925c663dd4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 34, 'total_tokens': 42, 'input_token_details': {'cache_read'